# 02 — Guardrails: Policy Engine & Risk Scoring

**Session: Month 5, Session 1**

## Learning Objectives
1. Understand why business rules should be code, not prompts
2. Use the Policy Engine to enforce declarative business rules
3. Apply the Risk Scorer to quantify action risk
4. Understand how policy + risk score together route to HITL

## The Problem with Prompt-Based Rules
If you put business rules in the system prompt:
- The LLM may ignore them under adversarial conditions
- Rules can 'drift' as the conversation context grows
- You can't unit-test a prompt rule
- Auditors can't verify what rules were active at transaction time

**Policy-as-Code** moves rules out of the LLM context into
deterministic Python that runs before every tool call.

In [ ]:
import sys
sys.path.insert(0, '../backend')

from app.guardrails.policy_engine import PolicyEngine, PolicyAction, Policy
from app.guardrails.risk_scorer import RiskScorer
import matplotlib.pyplot as plt
import numpy as np

## Part 1: Policy Engine

In [ ]:
engine = PolicyEngine()

scenarios = [
    ('Small refund ($25)', 'process_refund', {'amount': 25.0, 'order_id': 'ORD-1', 'reason': 'Wrong item'}),
    ('Medium refund ($200)', 'process_refund', {'amount': 200.0, 'order_id': 'ORD-1', 'reason': 'Damaged'}),
    ('Large refund ($750)', 'process_refund', {'amount': 750.0, 'order_id': 'ORD-1', 'reason': 'Damaged'}),
    ('Excessive refund ($1500)', 'process_refund', {'amount': 1500.0, 'order_id': 'ORD-1', 'reason': 'Test'}),
    ('Cancel delivered order', 'cancel_order', {'order_id': 'ORD-1', 'reason': 'Changed mind'},),
    ('Cancel processing order', 'cancel_order', {'order_id': 'ORD-2', 'reason': 'Changed mind'}),
    ('10k loyalty points', 'add_loyalty_points', {'customer_id': 'C-1', 'points': 10000, 'reason': 'Goodwill'}),
    ('Lookup order (read only)', 'lookup_order', {'order_id': 'ORD-1'}),
]

context = {'order_status': {'ORD-1': 'delivered', 'ORD-2': 'processing'}}

print(f'{"Scenario":<35} {"Action":<20} {"Triggered Policies"}')
print('=' * 80)
for name, tool, args in scenarios:
    result = engine.evaluate(tool, args, context)
    icon = {'allow': '✅', 'block': '🚫', 'require_approval': '⏸️', 'warn': '⚠️'}.get(result.action.value, '?')
    print(f'{name:<35} {icon} {result.action.value:<18} {result.triggered_policies}')

## Part 2: Risk Scoring

In [ ]:
scorer = RiskScorer(hitl_threshold=0.7)

tools_and_amounts = [
    ('lookup_order', {}),
    ('search_documents', {'query': 'return policy'}),
    ('update_shipping_address', {'order_id': 'ORD-1', 'new_address': '123 Main St'}),
    ('expedite_order_shipping', {'order_id': 'ORD-1'}),
    ('apply_discount_code', {'order_id': 'ORD-1', 'discount_code': 'SAVE20'}),
    ('add_loyalty_points', {'customer_id': 'C-1', 'points': 500}),
    ('cancel_order', {'order_id': 'ORD-1', 'reason': 'Customer request'}),
    ('process_refund', {'order_id': 'ORD-1', 'amount': 50}),
    ('process_refund', {'order_id': 'ORD-1', 'amount': 300}),
    ('process_refund', {'order_id': 'ORD-1', 'amount': 800}),
    ('process_refund', {'order_id': 'ORD-1', 'amount': 1200}),
]

print(f'{"Tool + Context":<50} {"Score":<8} {"Level":<12} {"HITL?"}')
print('=' * 80)
for tool, args in tools_and_amounts:
    s = scorer.score(tool, args)
    amount = args.get('amount', '')
    label = f'{tool}' + (f' (${amount})' if amount else '')
    hitl = '✅ YES' if s.requires_hitl else 'no'
    print(f'{label:<50} {s.score:<8.3f} {s.level:<12} {hitl}')

In [ ]:
# Visualise risk scores
tools = []
scores = []
colors = []

scorer2 = RiskScorer()
for tool, args in tools_and_amounts:
    s = scorer2.score(tool, args)
    amount = args.get('amount', '')
    label = tool + (f' ${amount}' if amount else '')
    tools.append(label)
    scores.append(s.score)
    color = {'none': 'green', 'low': 'yellowgreen', 'medium': 'orange', 'high': 'red', 'critical': 'darkred'}.get(s.level, 'gray')
    colors.append(color)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(tools, scores, color=colors, edgecolor='white')
ax.axvline(x=0.7, color='red', linestyle='--', label='HITL threshold (0.7)')
ax.set_xlabel('Risk Score')
ax.set_title('Agent Tool Risk Scores')
ax.set_xlim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()
print('Red bars = HITL approval required | Orange = high risk | Green = safe')

## Lab Exercise: Write Custom Policies

Add 3 new policies to the engine for these business rules:

1. **VIP Protection:** Gold/Platinum customers cannot have their loyalty points reduced
2. **Duplicate Refund Prevention:** Cannot issue a refund for the same order twice in one session
3. **Business Hours:** Cancellations over $200 require approval outside 9am-5pm EST

Each policy should:
- Have a clear name and description
- Have the correct action (BLOCK, WARN, or REQUIRE_APPROVAL)
- Have a unit test in `tests/test_guardrails.py`

In [ ]:
# TODO: Implement your custom policies here

def my_policy_condition(tool_name, args, context):
    # Replace this with your condition
    return False

my_policy = Policy(
    name='my_custom_policy',
    description='TODO: add description',
    condition=my_policy_condition,
    action=PolicyAction.WARN,
    priority=50,
    message='TODO: add message',
)

engine.add_policy(my_policy)
print(f'Policies registered: {len(engine.list_policies())}')